# Acceptance: lazy-idiom bridges

**This is a sign-off page, not a tutorial.** It exercises roadmap steps 31–35. Step 31 closes
composability finding `CX26`: a likelihood scale (`NormalLikelihood(sd=...)` and
`NegativeBinomial(dispersion=...)`) accepts any parameter-only rate expression, not only a
bare `Param`. The claim the reader checks is that an expression scores exactly like the float
it evaluates to. The automated checks live in one test file per section: section A in
`tests/test_epi_calibration.py`, section B in `tests/test_output_expr_dispatch.py`, section C in
`tests/test_output_defer.py`, section D in `tests/test_named_callables.py`, and section E in `tests/test_rate_children.py`.

Section B (roadmap step 32) closes composability finding `CX27`: NumPy functions such as `np.log`
and `np.minimum` apply to a named output (`outputs.ref(...)`), not only to a rate.

Section C (roadmap step 33) closes composability finding `CX28`: `defer(f)(outputs.ref(...), Param(...))`
applies an ordinary function to a solved output and a parameter.

Section D (roadmap step 34) closes composability finding `CX7`: `named(fn, "key")` gives a callable
hook a stable digest key, so a model rebuilt with a fresh function reuses its compiled program.

Section E (roadmap step 35) closes composability finding `CX29`: every pass over a rate
expression now walks the same children, so a `FlowRef` inside a `ForceOfInfection` is ordered,
and the force of infection's parameter-only children are hoisted.

The model is a seasonal SEIRS in two age groups, run for two years with weekly saves. Sections
B–E reuse this same model. Run every cell top to bottom (a few seconds). Sign off only if every **What to check** box holds.

| | You are checking that… |
|---|---|
| A | a negative-binomial `dispersion` written as an expression of a parameter scores exactly like the float it evaluates to, and the likelihood peaks near the true value |
| B | a named output takes the same functions as a rate (`np.log`, `np.minimum`, ...), and each expression **matches** the same function applied to the solved output |
| C | a function of a solved output and a parameter, `defer(delayed)(outputs.ref("incidence"), Param("delay"))`, gives a delayed, smoothed copy of incidence whose sum is close to incidence's, and a longer delay moves its peak later and lowers it |
| D | rebuilding the model five times with a fresh function retraces on every rebuild without `named(fn, "key")`, and once with it; the two models give the same derivative |
| E | a vaccination flow read inside the force of infection is computed **before** the infection flow, the model gives the same incidence with `hoist=True` and `hoist=False`, and the hoist table slots the force of infection's parameter-only part |

Roadmap step 31 added section A, step 32 section B, step 33 section C, step 34 section D and step 35 section E.

## The model: a seasonal SEIRS in two age groups

Two age groups mix assortatively. The infection rate is `beta * (1 + amp * sin(2π t / 365))`,
built as a rate expression of `Time()`, so the seasonal forcing is part of the model and not a
number that was precomputed. Recovered people lose immunity at rate `omega` (`R → S`, mean
duration 120 days). Without waning, the first wave burns out in about 40 of the 104 weeks and
leaves little signal for the dispersion. Incidence is the infection flow summed over age groups
and saved weekly with `FlowMass(...).total().midpoint()`.

In [ ]:
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    FlowMass,
    FlowModel,
    OutputSet,
    Param,
    Property,
    PropertyMap,
    SavePlan,
    Time,
    TransitionFlow,
)
from summer4.epi import FOIKind, ForceOfInfection, MixingMatrix
from summer4.epi.calibration import NegativeBinomial

state = Property("state", ("S", "E", "I", "R"))
age = Property("age", ("young", "old"))
pmap = PropertyMap.from_property(state).stratify(age)

beta = Param("beta") * (1 + Param("amp") * np.sin(2 * np.pi * Time() / 365))
mixing = MixingMatrix(
    age, np.array([[0.8, 0.2], [0.3, 0.7]]), normalize="rows", check_reciprocal=False
)
model = FlowModel(pmap)
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["E"],
        ForceOfInfection(
            "infection",
            infectious=state["I"],
            group_by=mixing.prop,
            mixing=mixing,
            kind=FOIKind.FREQUENCY,
            contact_rate=beta,
        ),
    )
)
model.add_flow(TransitionFlow("progression", state["E"], state["I"], Param("sigma")))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("gamma")))
model.add_flow(TransitionFlow("waning", state["R"], state["S"], Param("omega")))
compiled = model.compile()

y0 = np.zeros(pmap.size)
y0[pmap.select(state["S"] & age["young"])] = 79990.0
y0[pmap.select(state["I"] & age["young"])] = 10.0
y0[pmap.select(state["S"] & age["old"])] = 20000.0

TRUE_PARAMS = {"beta": 0.5, "amp": 0.3, "sigma": 0.2, "gamma": 1.0 / 7.0, "omega": 1.0 / 120.0}
weeks = np.arange(7.0, 729.0, 7.0)

outputs = OutputSet()
outputs["incidence"] = FlowMass("infection").total().midpoint()
solved = compiled.run(
    TRUE_PARAMS,
    y0,
    t0=0.0,
    t1=728.0,
    dt=0.25,
    save=outputs.plan(SavePlan(ts=weeks)),
    solver="euler",
)
named = outputs.evaluate(solved, TRUE_PARAMS)
predicted = np.asarray(named["incidence"].values).reshape(-1)

## A. A likelihood scale can be an expression

The **true** dispersion is `k = exp(log_k)` with `log_k = 1.5`. The first code cell draws the
observed weekly counts from a negative binomial around the model's incidence with a fixed seed.
The figure plots the model incidence and the noisy counts. The counts should scatter around the
model curve with a spread that grows with the level.

The second code cell builds `NegativeBinomial(dispersion=np.exp(Param("log_k")))` and evaluates
its `log_prob` at each `log_k` on a grid from 0 to 3. It does the same with a float dispersion,
`np.exp(log_k)`, computed at that grid point. The figure plots both curves.

In [ ]:
TRUE_LOG_K = 1.5
rng = np.random.default_rng(31)
k_true = float(np.exp(TRUE_LOG_K))
observed = rng.negative_binomial(k_true, k_true / (k_true + predicted)).astype(float)

weekly = pd.DataFrame(
    {"model incidence": predicted, "observed (negative-binomial draw)": observed},
    index=pd.Index(weeks, name="day"),
)
fig_data = weekly.plot(
    title="A. Weekly incidence: the model and the noisy observations",
    labels={"index": "day", "value": "infections per week"},
)
fig_data.show()

In [ ]:
grid = np.linspace(0.0, 3.0, 61)
expression_likelihood = NegativeBinomial(dispersion=np.exp(Param("log_k")))
expression_ll = []
float_ll = []
for log_k in grid:
    expression_ll.append(
        float(
            expression_likelihood.log_prob(
                jnp.asarray(observed), jnp.asarray(predicted), {"log_k": log_k}
            )
        )
    )
    float_likelihood = NegativeBinomial(dispersion=float(np.exp(log_k)))
    float_ll.append(
        float(float_likelihood.log_prob(jnp.asarray(observed), jnp.asarray(predicted), {}))
    )
expression_ll = np.asarray(expression_ll)
float_ll = np.asarray(float_ll)

curves = pd.DataFrame(
    {
        "Param expression: np.exp(Param('log_k'))": expression_ll,
        "float dispersion: np.exp(log_k)": float_ll,
    },
    index=pd.Index(grid, name="log_k"),
)
fig_ll = curves.plot(
    title="A. Log-likelihood against log_k: the expression and the float",
    labels={"index": "log_k", "value": "mean log-likelihood per week"},
)
fig_ll.data[1].update(line={"color": "black", "dash": "dot", "width": 2})
fig_ll.add_vline(x=TRUE_LOG_K, line_dash="dash", annotation_text="true log_k")
fig_ll.show()

pd.Series(
    {
        "true log_k": TRUE_LOG_K,
        "grid maximum log_k": round(float(grid[np.argmax(expression_ll)]), 2),
        "max |expression - float| over the grid": float(
            np.max(np.abs(expression_ll - float_ll))
        ),
    },
    name="A",
    dtype=object,
)

> **What to check**
>
> - In the first figure, the noisy observations scatter around the model incidence curve, and
>   the scatter widens where the incidence is high.
> - In the second figure, the solid curve (dispersion written as `np.exp(Param("log_k"))`) and
>   the dotted curve (a float `np.exp(log_k)` at each point) lie **on top of each other** across
>   the whole grid.
> - The maximum of the log-likelihood lies **within about 0.3 of the true `log_k` = 1.5**
>   (the dashed line). It is a single weekly series, so the peak is noisy; across seeds it
>   lands within 0.3 for most draws, not all.
> - The table's "max |expression - float|" is at float32 rounding (far below 1e-4).

## B. Functions apply to named outputs

Named outputs take the same functions as rate expressions: NumPy ufuncs such as `np.log` and
`np.minimum`. The claim the reader checks is that a
function written on `outputs.ref(...)` gives the same series as the same function applied to
the solved output directly.

The first code cell defines two more named outputs on the model above. `incidence` is the
weekly incidence from the model cell and is not redefined. `log_incidence` is
`np.log(outputs.ref("incidence"))`, and `capped` is `np.minimum(outputs.ref("incidence"), CAP)`,
where `CAP` is half the peak of the model incidence. Neither adds a save, because both read the
same leaf. The cell then applies `np.log` and `np.minimum` directly to the evaluated
`incidence` output.

The next two cells plot each named output as a solid line, with the direct computation as a
dashed black line on top of it, and print a small table of the differences.

In [ ]:
CAP = 0.5 * float(predicted.max())

outputs["log_incidence"] = np.log(outputs.ref("incidence"))
outputs["capped"] = np.minimum(outputs.ref("incidence"), CAP)
named_b = outputs.evaluate(solved, TRUE_PARAMS)

direct_log = np.log(named_b["incidence"])
direct_capped = np.minimum(named_b["incidence"], CAP)

In [ ]:
incidence_b = np.asarray(named_b["incidence"].values).reshape(-1)
log_named = np.asarray(named_b["log_incidence"].values).reshape(-1)
log_direct = np.asarray(direct_log.values).reshape(-1)

log_plot = pd.DataFrame(
    {
        "log_incidence (named output)": log_named,
        "np.log(incidence), applied directly": log_direct,
    },
    index=pd.Index(weeks, name="day"),
)
fig_log = log_plot.plot(
    title="B. Log of incidence: the named output and the direct computation",
    labels={"index": "day", "value": "log(infections per week)"},
)
fig_log.data[1].update(line={"color": "black", "dash": "dash", "width": 2})
fig_log.show()

In [ ]:
capped_named = np.asarray(named_b["capped"].values).reshape(-1)
capped_direct = np.asarray(direct_capped.values).reshape(-1)

cap_plot = pd.DataFrame(
    {
        "incidence (solved)": incidence_b,
        "capped (named output)": capped_named,
        "np.minimum(incidence, CAP), applied directly": capped_direct,
    },
    index=pd.Index(weeks, name="day"),
)
fig_cap = cap_plot.plot(
    title=f"B. Incidence capped at {CAP:.1f} per week",
    labels={"index": "day", "value": "infections per week"},
)
fig_cap.data[0].update(line={"color": "lightgrey", "width": 2})
fig_cap.data[2].update(line={"color": "black", "dash": "dash", "width": 2})
fig_cap.add_hline(y=CAP, line_dash="dot", annotation_text="cap")
fig_cap.show()

pd.Series(
    {
        "CAP (half the peak incidence)": round(CAP, 2),
        "max |log_incidence - np.log(incidence)|": float(np.max(np.abs(log_named - log_direct))),
        "max |capped - np.minimum(incidence, CAP)|": float(
            np.max(np.abs(capped_named - capped_direct))
        ),
        "weeks where incidence exceeds CAP": int(np.sum(incidence_b > CAP)),
        "weeks where capped sits at CAP": int(np.sum(np.isclose(capped_named, CAP))),
    },
    name="B",
    dtype=object,
)

> **What to check**
>
> - In the first figure, the solid line (`log_incidence`, a named output) lies **on top of**
>   the dashed black line (`np.log` applied directly to the solved incidence) across the whole
>   two years.
> - In the second figure, the solid coloured `capped` line lies on the dashed black
>   `np.minimum` line everywhere. It is **flat at the cap** (the dotted line) in every week
>   where the grey incidence is above the cap, and follows the grey incidence in the other
>   weeks.
> - In the table, both "max |...|" differences are zero or at float32 rounding (below 1e-4).
>   The count of weeks where incidence exceeds CAP equals the count of weeks where capped
>   sits at CAP.

## C. A function of an output and a parameter

summer2's `Function` applies an ordinary Python function to a solved output and to parameters.
`defer(f)(outputs.ref("incidence"), Param("delay"))` does the same, lazily, inside an
`OutputSet`. Here `f` is `delayed`: it convolves weekly incidence with a gamma-shaped delay whose
mean is `delay` days, so the result is a notification series, a delayed and smoothed copy of
incidence.

The first code cell defines `delayed`. It takes an `Output` and a parameter, and returns an array
of the same length. The second cell defines the notifications set. It repeats the `incidence`
leaf defined in the model cell so this section runs on its own, then evaluates the notifications for three
delays (3, 10 and 21 days) on the solved run.

The figure plots incidence (black) and the three notification curves. The table gives each curve's
sum, its sum relative to incidence, and its peak.

In [ ]:
from jax.scipy.stats import gamma as jax_gamma

from summer4 import PropertyData, defer

GAMMA_SHAPE = 4.0
KERNEL_WEEKS = 26


def delayed(series, delay):
    """Convolve a weekly series with a gamma kernel whose mean is ``delay`` days."""
    values = series.values.data if isinstance(series.values, PropertyData) else series.values
    x = jnp.asarray(values)
    edges = jnp.arange(KERNEL_WEEKS + 1, dtype=jnp.float32)
    cdf = jax_gamma.cdf(edges, GAMMA_SHAPE, scale=delay / 7.0 / GAMMA_SHAPE)
    weights = jnp.diff(cdf)
    kernel = weights / jnp.sum(weights)
    return jnp.convolve(x, kernel, mode="full")[: x.shape[0]]

In [ ]:
DELAYS = (3.0, 10.0, 21.0)

notif_outputs = OutputSet()
notif_outputs["incidence"] = FlowMass("infection").total().midpoint()
notif_outputs["notifications"] = defer(delayed)(notif_outputs.ref("incidence"), Param("delay"))

notif_curves = {}
for delay in DELAYS:
    evaluated = notif_outputs.evaluate(solved, {**TRUE_PARAMS, "delay": delay})
    notif_curves[delay] = np.asarray(evaluated["notifications"].values).reshape(-1)
incidence_c = np.asarray(evaluated["incidence"].values).reshape(-1)

curves_c = pd.DataFrame(
    {"incidence (solved)": incidence_c}
    | {f"notifications, delay {delay:g} days": notif_curves[delay] for delay in DELAYS},
    index=pd.Index(weeks, name="day"),
)
fig_c = curves_c.plot(
    title="C. Incidence and its notifications at three delays",
    labels={"index": "day", "value": "infections or notifications per week"},
)
fig_c.data[0].update(line={"color": "black", "width": 2})
fig_c.show()

In [ ]:
summary_rows = [("incidence (solved)", incidence_c)] + [
    (f"notifications, delay {delay:g} days", notif_curves[delay]) for delay in DELAYS
]
summary_c = pd.DataFrame(
    [
        {
            "curve": name,
            "sum": float(series.sum()),
            "sum / incidence sum": float(series.sum() / incidence_c.sum()),
            "peak day": int(weeks[np.argmax(series)]),
            "peak per week": float(series.max()),
        }
        for name, series in summary_rows
    ]
).set_index("curve")
summary_c

> **What to check**
>
> - In the figure, each notification curve is a delayed and smoothed copy of incidence. It
>   follows the black incidence curve and has lower peaks. The longer delays (10 and 21 days)
>   peak later than incidence. The 3-day curve peaks in the same week as incidence, because
>   saves are weekly and its shift is under one save interval.
> - The longer the delay, the later the peak and the lower the peak. In the table, the peak day
>   does not move earlier, and moves later for 10 and 21 days, and the peak per week falls as
>   the delay grows from 3 to 21 days.
> - The kernel sums to 1 and has no negative weights, so no notification sum can exceed the
>   incidence sum. Each sum in the table is close to the incidence sum (within about 4%). The
>   small shortfall is notifications that fall after the last week: incidence is still rising at
>   two years, so some of its late mass is delayed past the end of the series.

## D. Rebuilding a model with a fresh function

A compiled model is reused, but a notebook cell or a scenario sweep often rebuilds it. This
section rebuilds the SEIRS model from the model cell five times. Each rebuild adds a `Transform`
on recovery that multiplies it by `1 + 0.1 sin(t / 30)`, and each rebuild creates a new closure for
that function. By default a closure is keyed by its identity, so every rebuild has a new jit-cache
key and retraces. The second case passes the same function through `named(fn, "seasonal-recovery")`,
which keys the digest on the name.

The vector field of each rebuild is evaluated once under `jax.jit`, with the model as a static
argument. A trace is counted each time the jitted function's Python body runs, so a cache
hit counts zero. The first code cell
defines the builder and checks that one rebuild gives the same derivative with and without `named`.
The second cell counts traces for five rebuilds in each case and plots them.


In [ ]:
import functools
from typing import Any

import jax

from summer4 import CompiledModel, PropertyData, Transform, named

SEASON_DAYS = 30.0
REBUILDS = 5


def build_seirs(name: str | None = None) -> CompiledModel:
    """Rebuild the SEIRS model of the model cell, with a fresh recovery ``Transform``.

    Every call creates a new closure. ``name=None`` keys its digest by identity, and a string
    wraps the closure in ``named``.
    """

    def seasonal_recovery(prev: Any, t: Any) -> Any:
        return prev * (1.0 + 0.1 * jnp.sin(t / SEASON_DAYS))

    adjust_fn = seasonal_recovery if name is None else named(seasonal_recovery, name)
    rebuilt = FlowModel(pmap)
    rebuilt.add_flow(
        TransitionFlow(
            "infection",
            state["S"],
            state["E"],
            ForceOfInfection(
                "infection",
                infectious=state["I"],
                group_by=mixing.prop,
                mixing=mixing,
                kind=FOIKind.FREQUENCY,
                contact_rate=beta,
            ),
        )
    )
    rebuilt.add_flow(TransitionFlow("progression", state["E"], state["I"], Param("sigma")))
    rebuilt.add_flow(
        TransitionFlow(
            "recovery",
            state["I"],
            state["R"],
            Param("gamma"),
            adjust=(Transform(adjust_fn, Time()),),
        )
    )
    rebuilt.add_flow(TransitionFlow("waning", state["R"], state["S"], Param("omega")))
    return rebuilt.compile()


y_arr = PropertyData.wrap(pmap, y0)
dy_bare = build_seirs(None).vector_field(0.0, y_arr, TRUE_PARAMS)
dy_named = build_seirs("seasonal-recovery").vector_field(0.0, y_arr, TRUE_PARAMS)
pd.Series(
    {
        "max |dy named - dy bare|": float(
            np.max(np.abs(np.asarray(dy_named.data) - np.asarray(dy_bare.data)))
        ),
    },
    name="D",
    dtype=object,
)


In [ ]:
def retraces(name: str | None) -> int:
    """Rebuild the model ``REBUILDS`` times, run each once under jit, and count the traces."""
    count = 0

    @functools.partial(jax.jit, static_argnums=0)
    def run_once(cm: CompiledModel, state_y: PropertyData) -> Any:
        nonlocal count
        count += 1
        return cm.vector_field(0.0, state_y, TRUE_PARAMS)

    # Hold every rebuilt model until the loop ends: a freed closure's id can be reused.
    rebuilds = [build_seirs(name) for _ in range(REBUILDS)]
    for cm in rebuilds:
        run_once(cm, y_arr)
    return count


counts = pd.Series(
    {
        "bare closures": retraces(None),
        'named(fn, "seasonal-recovery")': retraces("seasonal-recovery"),
    },
    name="retraces per 5 rebuilds",
)
fig_d = counts.plot.bar(
    title="D. Retraces when the model is rebuilt five times",
    labels={"index": "how the Transform is keyed", "value": "traces"},
)
fig_d.show()
counts


> **What to check**
>
> - The derivative row in the first table is zero: the model with `named` and the model with a
>   bare closure give the same vector field, so `named` changes the cache key and nothing else.
> - In the bar plot, the bare-closure bar is **5**. Every rebuild retraces, because each fresh
>   closure has a new identity.
> - The `named` bar is **1**. The first rebuild traces, and the other four hit the cache.
> - `named` is a promise: every function given the same name must compute the same program. The
>   recovery adjustment is identical on every rebuild here, so the promise holds.


## E. A flow reference inside the force of infection is ordered

A force of infection can read another flow. Here the SEIRS model gains a vaccination flow
`S → R` at rate `nu`, and the contact rate is scaled by the total vaccination flow:
`beta(t) * (1 - k * vaccination)`, with `k = vax_effect / pop_scale`. That scaling reads
`FlowRef("vaccination")` inside `ForceOfInfection.contact_rate`. The infection flow is added to
the model first on purpose, so only a correct ordering puts vaccination before it.

The first code cell builds the model and displays the compiled flow order. The second cell runs
it twice, with `hoist=True` and with `hoist=False`, and overlays the weekly incidence. It adds a
third line with `vax_effect = 0`, where the force of infection ignores vaccination. The third cell
displays the hoist table: the run-start values the compiled model computes once. `k` depends
on parameters only, so it is slotted even though it sits inside the force of infection.

In [ ]:
from summer4 import FlowRef

E_PARAMS = {**TRUE_PARAMS, "nu": 0.002, "vax_effect": 0.5, "pop_scale": 1000.0}

k_vax = Param("vax_effect") / Param("pop_scale")  # parameters only: hoistable
beta_vax = beta * (1.0 - k_vax * FlowRef("vaccination").sum())


def build_vaccination_model(*, hoist: bool) -> CompiledModel:
    """SEIRS with a vaccination flow that scales the force of infection."""
    vaccinated = FlowModel(pmap)
    vaccinated.add_flow(
        TransitionFlow(
            "infection",
            state["S"],
            state["E"],
            ForceOfInfection(
                "infection",
                infectious=state["I"],
                group_by=mixing.prop,
                mixing=mixing,
                kind=FOIKind.FREQUENCY,
                contact_rate=beta_vax,
            ),
        )
    )
    vaccinated.add_flow(TransitionFlow("vaccination", state["S"], state["R"], Param("nu")))
    vaccinated.add_flow(TransitionFlow("progression", state["E"], state["I"], Param("sigma")))
    vaccinated.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("gamma")))
    vaccinated.add_flow(TransitionFlow("waning", state["R"], state["S"], Param("omega")))
    return vaccinated.compile(hoist=hoist)


compiled_e = build_vaccination_model(hoist=True)
pd.DataFrame(
    {"position": range(len(compiled_e.order)), "flow": compiled_e.order}
).set_index("position")

In [ ]:
compiled_e_off = build_vaccination_model(hoist=False)
outputs_e = OutputSet()
outputs_e["incidence"] = FlowMass("infection").total().midpoint()
plan_e = outputs_e.plan(SavePlan(ts=weeks))


def weekly_incidence(cm: CompiledModel, params: dict) -> np.ndarray:
    result = cm.run(params, y0, t0=0.0, t1=728.0, dt=0.25, save=plan_e, solver="euler")
    return np.asarray(outputs_e.evaluate(result, params)["incidence"].values).reshape(-1)


incidence_on = weekly_incidence(compiled_e, E_PARAMS)
incidence_off = weekly_incidence(compiled_e_off, E_PARAMS)
incidence_novax = weekly_incidence(compiled_e, {**E_PARAMS, "vax_effect": 0.0})

curves_e = pd.DataFrame(
    {
        "hoist=True": incidence_on,
        "hoist=False": incidence_off,
        "vax_effect = 0 (vaccination ignored)": incidence_novax,
    },
    index=pd.Index(weeks, name="day"),
)
fig_e = curves_e.plot(
    title="E. Incidence with a vaccination flow inside the force of infection",
    labels={"index": "day", "value": "infections per week"},
)
fig_e.data[1].update(line={"color": "black", "dash": "dot", "width": 2})
fig_e.show()

pd.Series(
    {
        "max |hoist=True - hoist=False|": float(np.max(np.abs(incidence_on - incidence_off))),
        "peak incidence with vaccination in the FOI": float(incidence_on.max()),
        "peak incidence with vax_effect = 0": float(incidence_novax.max()),
    },
    name="E",
    dtype=object,
)

In [ ]:
hoist_entries = compiled_e.hoist_table.entries
pd.DataFrame(
    [
        {"slotted node": type(entry.node).__name__, "part": entry.part, "node": repr(entry.node)}
        for entry in hoist_entries
    ]
)

> **What to check**
>
> - In the flow-order table, `vaccination` comes **before** `infection`, although `infection` was
>   added to the model first. That is the ordering the `FlowRef` inside the force of infection
>   asks for.
> - In the figure, the solid `hoist=True` line and the dotted black `hoist=False` line lie **on
>   top of each other** everywhere. The "max |hoist=True - hoist=False|" row is zero or at float32
>   rounding (below 1e-3).
> - The third line (`vax_effect = 0`) peaks **higher** than the other two. With vaccination
>   removed from the contact rate, the epidemic is larger, so the flow reference is being read.
> - The hoist table has an entry for the `k = vax_effect / pop_scale` division: a `BinOp` with
>   part `value`, built from `FieldRef`s only. It sits inside the force of infection's contact
>   rate, which the hoist table did not reach before step 35.
> - The hoist table lists no entry for the contact rate as a whole or for the force of
>   infection. They read `FlowRef("vaccination")` and the state, so they run every step.